# 05 上传包：远端一致性验证

本 Notebook 是独立上传包的唯一主入口。它只把现货文件交给非零退出、V156 大跌和 V189 大涨三个冻结引擎；三状态文件在引擎完成后才作为审计基准读取。运行结束后以 `runtime_outputs/最终一致性结论.json` 为最终结论。

In [ ]:
from pathlib import Path
import sys

def find_package_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'remote_validation.py').is_file():
            return candidate
    raise FileNotFoundError('没有找到 05_上传包 根目录')

PACKAGE_ROOT = find_package_root(Path.cwd().resolve())
sys.path.insert(0, str(PACKAGE_ROOT / 'src'))
from remote_validation import run_remote_validation

manifest = run_remote_validation()
manifest

In [ ]:
import json
import pandas as pd

result_path = PACKAGE_ROOT / 'runtime_outputs' / '最终一致性结论.json'
result = json.loads(result_path.read_text(encoding='utf-8'))
print('success =', result['success'])
print('非零五列 =', result['nonzero_audit']['no_date_or_value_difference'])
print('三状态审计 =', result['state_audit']['no_date_or_value_difference'])
print('大跌逐日 =', result['extreme_audits']['down']['no_date_or_value_difference'])
print('大涨逐日 =', result['extreme_audits']['up']['no_date_or_value_difference'])
execution_path = PACKAGE_ROOT / 'runtime_outputs' / '最终执行日简表.csv'
execution = pd.read_csv(execution_path, encoding='utf-8-sig')
print('最终执行日简表 =', execution_path)
print('列 =', list(execution.columns), '；行数 =', len(execution))
display(execution.head(20))